# Replicates — four independent starts at 8 assets on `ibm_fez`

**Dr. Muhammad Faryad** · follow-up to `NB16b_qaoa_jobs_8_assets.ipynb`

The main study ran each arm once per instance size. This notebook repeats the 8-asset comparison
from **three further starting points**, so that the headline claim — batched pattern search reaches
good parameters in one or two jobs, the serial optimiser needs five to eight — is reported with a
spread rather than as a single trajectory.

Everything is identical to `NB16b`: same instance, same $p = 3$, same 1,024 shots per candidate,
same 10 jobs per arm, same 14,336 shots per job for both arms, same CVaR$_{0.5}$, same initial
radius. **Only the starting point changes**, and within a start both arms get the *same* one.

The starts are the linear ramp at four scales $\Delta t \in \{0.35, 0.5, 0.6, 0.85\}$;
$\Delta t = 0.5$ is the run already in hand, so this notebook contributes the other three and the
paper reports four.

Cost: 3 starts × (10 + 10 optimisation jobs + 2 read-outs) = **66 jobs ≈ 9 minutes of metered QPU**.
Wall clock will be far longer — budget an hour, and note that the notebook **writes its JSON after
every start**, so stopping after start 1 or 2 still leaves usable data.

In [1]:
import json, time, warnings
from math import comb

import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

warnings.filterwarnings("ignore")

ACCOUNT_NAME = "Faryad-free-ibm"
BACKEND_NAME = "ibm_fez"

N_ASSETS = 8            # everything below matches NB16b exactly
P_LAYERS = 3
SHOTS    = 1024
N_JOBS   = 10
ALPHA    = 0.5
RADIUS0  = 0.25
SHOTS_FINAL = 8192

DT_STARTS = [0.35, 0.60, 0.85]     # dt = 0.50 is the run already in hand
QPU_BUDGET_S = 700                 # generous: the model predicts ~536 s

N_CAND    = 2 + 4 * P_LAYERS
JOB_SHOTS = N_CAND * SHOTS
N_STATES_CACHE = {N_ASSETS: 2 ** N_ASSETS}

print(f"{N_ASSETS} assets, choose {N_ASSETS//2}, p = {P_LAYERS}")
print(f"{len(DT_STARTS)} starts x ({2*N_JOBS} optimisation + 2 read-out) jobs "
      f"= {len(DT_STARTS)*(2*N_JOBS+2)} jobs")

8 assets, choose 4, p = 3
3 starts x (20 optimisation + 2 read-out) jobs = 66 jobs


---
## The instance, the circuit and the chain

Identical to `NB16b`. The assert on the Ising mapping runs over all $2^8$ bitstrings, and the chain is re-selected from the calibration of the day, so it need not be the same physical line as the original run — that variation is part of what these replicates measure.

In [2]:
asset_names = ['Google', 'Intel', 'Nvidia', 'Goldman Sachs', 'ExxonMobil', 'IBM',
               'JPMorgan', 'Bank of America', 'Johnson & Johnson', 'Pfizer',
               'Walmart', 'Amazon']
tickers = ['GOOG', 'INTC', 'NVDA', 'GS', 'XOM', 'IBM', 'JPM', 'BAC', 'JNJ', 'PFE', 'WMT', 'AMZN']

expected_returns_12 = np.array(
    [59.14, 168.17, 28.48, 41.97, 48.07, 12.59, 24.56, 31.27, 46.84, 20.29, 14.46, 23.06])

covariance_12 = np.array(
    [[10.41, 4.95, 2.92, 3.04, -2.16, 1.20, 1.36, 1.29, -0.16, 0.51, 0.21, 5.63],
     [4.95, 62.12, 10.62, 8.27, -3.19, -2.50, 1.97, 1.80, -2.81, -0.11, -1.29, 4.88],
     [2.92, 10.62, 13.48, 5.41, -1.81, 0.12, 1.52, 1.06, -1.55, -0.51, -1.67, 3.69],
     [3.04, 8.27, 5.41, 9.88, -1.54, 0.52, 4.54, 3.72, -0.88, 0.41, -0.84, 2.51],
     [-2.16, -3.19, -1.81, -1.54, 6.51, -0.71, -0.36, -0.43, 0.52, -0.18, 0.41, -1.97],
     [1.20, -2.50, 0.12, 0.52, -0.71, 23.34, 1.66, 2.14, 0.15, 1.70, -1.17, 2.05],
     [1.36, 1.97, 1.52, 4.54, -0.36, 1.66, 4.97, 3.57, 0.09, 0.56, -0.28, 1.27],
     [1.29, 1.80, 1.06, 3.72, -0.43, 2.14, 3.57, 4.50, -0.13, 0.50, 0.08, 1.32],
     [-0.16, -2.81, -1.55, -0.88, 0.52, 0.15, 0.09, -0.13, 3.44, 1.29, 1.37, -0.94],
     [0.51, -0.11, -0.51, 0.41, -0.18, 1.70, 0.56, 0.50, 1.29, 5.58, 0.23, 0.54],
     [0.21, -1.29, -1.67, -0.84, 0.41, -1.17, -0.28, 0.08, 1.37, 0.23, 5.98, -0.89],
     [5.63, 4.88, 3.69, 2.51, -1.97, 2.05, 1.27, 1.32, -0.94, 0.54, -0.89, 11.84]])

LAMBDAS = {6: 1.771087, 8: 1.652797, 10: 1.553166, 12: 1.360032}   # as in the four main runs
MUS     = {6: 8.0,      8: 9.0,      10: 12.0,     12: 13.0}


def build_instance(n):
    """Everything classical about the n-asset instance, exactly as in the main runs."""
    B, lam, mu = n // 2, LAMBDAS[n], MUS[n]
    r = expected_returns_12[:n]
    C = covariance_12[:n, :n]
    N = 2 ** n
    bits = (np.arange(N)[:, None] >> np.arange(n)[None, :]) & 1
    cost = (lam * np.einsum("ki,ij,kj->k", bits, C, bits) - bits @ r
            + mu * (bits.sum(1) - B) ** 2)
    feas = bits.sum(1) == B

    Q = lam * C + mu * np.ones((n, n))
    L = r + 2 * mu * B * np.ones(n)
    offset = mu * B ** 2
    h = np.zeros(n)
    J = np.zeros((n, n))
    for i in range(n):
        offset += Q[i, i] / 2 - L[i] / 2
        h[i] += -Q[i, i] / 2 + L[i] / 2
    for i in range(n):
        for j in range(i + 1, n):
            cij = 2 * Q[i, j]
            offset += cij / 4
            h[i] -= cij / 4
            h[j] -= cij / 4
            J[i, j] += cij / 4
    scale = max(np.abs(h).max(), np.abs(J).max())
    h, J = h / scale, J / scale
    z = 1 - 2 * bits
    diag = z @ h
    for i in range(n):
        for j in range(i + 1, n):
            diag += J[i, j] * z[:, i] * z[:, j]
    assert np.abs(cost - (scale * diag + offset)).max() < 1e-8, "Ising mapping is wrong"

    return dict(n=n, B=B, lam=lam, mu=mu, names=[tickers[i] for i in range(n)],
                r=r, C=C, N=N, bits=bits, cost=cost, feas=feas, h=h, J=J,
                diag=diag, scale=float(scale), offset=float(offset),
                order=np.argsort(diag), C_opt=float(cost.min()),
                opt_index=int(cost.argmin()), C_worst=float(cost[feas].max()),
                C_mean_feas=float(cost[feas].mean()))


def qaoa_probs(inst, params, reps):
    """Noiseless QAOA distribution; params = [beta_0..., gamma_0...]."""
    n, N, diag = inst["n"], inst["N"], inst["diag"]
    betas, gammas = params[:reps], params[reps:]
    psi = np.ones(N, dtype=complex) / np.sqrt(N)
    for k in range(reps):
        psi = psi * np.exp(-1j * gammas[k] * diag)
        psi = psi.reshape([2] * n)
        cb, sb = np.cos(betas[k]), -1j * np.sin(betas[k])
        for q in range(n):
            ax = n - 1 - q
            psi = np.moveaxis(psi, ax, 0)
            a, b = psi[0].copy(), psi[1].copy()
            psi[0], psi[1] = cb * a + sb * b, sb * a + cb * b
            psi = np.moveaxis(psi, 0, ax)
        psi = psi.reshape(-1)
    return np.abs(psi) ** 2


def cvar(inst, probs, alpha):
    o = inst["order"]
    e, p = inst["diag"][o], probs[o]
    cum = np.cumsum(p)
    k = min(int(np.searchsorted(cum, alpha)) + 1, p.size)
    w = p[:k].copy()
    w[-1] -= max(cum[k - 1] - alpha, 0.0)
    return float(e[:k] @ w / w.sum()) if w.sum() > 0 else float(e[0])


def quality(inst, probs):
    feas, cost = inst["feas"], inst["cost"]
    fm = float(probs[feas].sum())
    cond = probs * feas / max(fm, 1e-15)
    mcf = float(cost @ cond)
    return dict(feas_mass=fm, mean_cost=float(cost @ probs), mean_cost_feas=mcf,
                ratio=float((inst["C_worst"] - mcf) / (inst["C_worst"] - inst["C_opt"])),
                p_opt=float(probs[inst["opt_index"]]),
                p_opt_feas=float(cond[inst["opt_index"]]))


def qaoa_swap_network(inst, reps):
    """QAOA on a linear chain; complete ZZ graph via an odd-even transposition network."""
    h, J, n = inst["h"], inst["J"], inst["n"]
    betas, gammas = ParameterVector("b", reps), ParameterVector("g", reps)
    qc = QuantumCircuit(QuantumRegister(n, "q"), ClassicalRegister(n, "c"))
    qc.h(range(n))
    perm = list(range(n))
    for layer in range(reps):
        g, be = gammas[layer], betas[layer]
        for pos in range(n):
            if h[perm[pos]]:
                qc.rz(2 * g * h[perm[pos]], pos)
        for step in range(n):
            for pos in range(step % 2, n - 1, 2):
                a, b = perm[pos], perm[pos + 1]
                w = J[min(a, b), max(a, b)]
                qc.cx(pos, pos + 1)                 # RZZ(2 g w) . SWAP in 3 CX
                qc.rz(2 * g * w, pos + 1)
                qc.cx(pos + 1, pos)
                qc.cx(pos, pos + 1)
                perm[pos], perm[pos + 1] = b, a
        qc.rx(2 * be, range(n))
    for pos in range(n):
        qc.measure(pos, perm[pos])                  # clbit index == logical qubit index
    return qc, perm


def best_chain(backend, n, beam=3000):
    """Lowest-total-error simple path of n physical qubits."""
    t = backend.target
    cz = {e: (p.error if p and p.error is not None else 0.01) for e, p in t["cz"].items()}
    ro = {q[0]: (p.error if p and p.error is not None else 0.02) for q, p in t["measure"].items()}
    adj = {}
    for (a, b), e in cz.items():
        adj.setdefault(a, []).append((b, e))
    w = lambda e: -np.log(max(1e-6, 1 - e))
    paths = [([q], w(ro.get(q, 0.02))) for q in range(backend.num_qubits)]
    for _ in range(n - 1):
        nxt = []
        for path, c in paths:
            for b, e in adj.get(path[-1], []):
                if b not in path:
                    nxt.append((path + [b], c + w(e) + w(ro.get(b, 0.02))))
        nxt.sort(key=lambda t: t[1])
        seen, keep = set(), []
        for path, c in nxt:
            key = (path[-1], frozenset(path))
            if key not in seen:
                seen.add(key)
                keep.append((path, c))
            if len(keep) >= beam:
                break
        paths = keep
    return min(paths, key=lambda t: t[1])

In [3]:
# ============================================================
# The cost model, calibrated on the 102 jobs of the first study
#     qpu_seconds(job) = 3.00 s + (352.6 + 0.267 * D2q) us * shots
# reproduces every job of the four main runs to better than 0.05 s.
# ============================================================
JOB_FIXED_S = 3.00
T_SHOT = lambda d2q: 1e-6 * (352.6 + 0.2673 * d2q)

n_jobs_used, shots_used, qpu_used = 0, 0, 0.0
job_log = []


def predict_qpu(shots, d2q):
    return JOB_FIXED_S + T_SHOT(d2q) * shots


class BudgetStop(Exception):
    """Raised instead of overrunning QPU_BUDGET_S, so the notebook always saves its data."""


def submit(isa, params, shots, reps, n_states, d2q, tag, reserve=0.0):
    """ONE job, ONE PUB, len(params) parameter rows."""
    global n_jobs_used, shots_used, qpu_used
    params = np.atleast_2d(np.asarray(params, float))
    rows, total = params.shape[0], params.shape[0] * shots
    predicted = predict_qpu(total, d2q)
    if qpu_used + predicted + reserve > QPU_BUDGET_S:
        raise BudgetStop(f"'{tag}' would need ~{predicted:.1f} s on top of {qpu_used:.1f} s "
                         f"(+{reserve:.1f} s reserved); cap is {QPU_BUDGET_S:.0f} s")
    t0 = time.time()
    pub = np.zeros_like(params)
    order = [p.name for p in isa.parameters]
    for i, nm in enumerate([f"b[{k}]" for k in range(reps)] + [f"g[{k}]" for k in range(reps)]):
        pub[:, order.index(nm)] = params[:, i]
    job = sampler.run([(isa, pub)], shots=shots)
    data = job.result()[0].data.c
    dists = []
    for k in range(rows):                       # never call get_int_counts() unindexed
        cnt = data[k].get_int_counts()
        d_ = np.zeros(n_states)
        d_[list(cnt.keys())] = np.array(list(cnt.values())) / shots
        dists.append(d_)
    try:
        measured = float(job.metrics()["usage"]["quantum_seconds"])
    except Exception:
        measured = predicted
    n_jobs_used += 1
    shots_used += total
    qpu_used += measured
    job_log.append(dict(tag=tag, job=n_jobs_used, id=job.job_id(), rows=rows, shots=shots,
                        reps=reps, d2q=d2q, qpu_predicted=predicted, qpu_measured=measured,
                        wall=time.time() - t0, params=params.tolist(),
                        counts=[{int(i): int(round(x[i] * shots)) for i in np.nonzero(x)[0]}
                                for x in dists]))
    print(f"    job {n_jobs_used:3d} [{tag}] id={job.job_id()}  {measured:5.2f} qpu s "
          f"(model {predicted:5.2f})  {time.time()-t0:5.0f} s wall  "
          f"(total {qpu_used:6.1f} / {QPU_BUDGET_S:.0f} s)")
    return dists

In [4]:
inst = build_instance(N_ASSETS)
service = QiskitRuntimeService(name=ACCOUNT_NAME)
backend = service.backend(BACKEND_NAME)
print(f"{backend.name}: {backend.status().pending_jobs} jobs queued")

chain, chain_cost = best_chain(backend, N_ASSETS)
circuit, perm = qaoa_swap_network(inst, P_LAYERS)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend,
                                  initial_layout=chain, seed_transpiler=42)
isa_circuit = pm.run(circuit)
used = sorted({isa_circuit.find_bit(q).index for i in isa_circuit.data for q in i.qubits
               if i.operation.name in ("cz", "measure")})
assert set(used) <= set(chain), "the pass manager moved the layout"

D2Q = isa_circuit.depth(lambda i: i.operation.name == "cz")
n2q = sum(1 for i in isa_circuit.data if i.operation.name == "cz")
print(f"chain {chain}")
print(f"{n2q} two-qubit gates, two-qubit depth {D2Q}")
print(f"optimum: {[inst['names'][i] for i in range(N_ASSETS) if inst['bits'][inst['opt_index'], i]]}"
      f"  cost {inst['C_opt']:.2f}")

sampler = SamplerV2(mode=backend)
sampler.options.dynamical_decoupling.enable = True
sampler.options.twirling.enable_gates = True
sampler.options.twirling.enable_measure = True
print(f"predicted: {predict_qpu(JOB_SHOTS, D2Q):.2f} s per optimisation job, "
      f"{len(DT_STARTS)*(2*N_JOBS*predict_qpu(JOB_SHOTS, D2Q) + 2*predict_qpu(SHOTS_FINAL, D2Q)):.0f} s total")

ibm_fez: 1 jobs queued
chain [141, 142, 143, 136, 123, 124, 125, 117]
252 two-qubit gates, two-qubit depth 72
optimum: ['GOOG', 'INTC', 'XOM', 'BAC']  cost -161.10
predicted: 8.33 s per optimisation job, 536 s total


---
## The two arms

Unchanged from the main study, wrapped in a function so they can be run once per start. Both arms are given the same `x0`, the same job count and the same shots per job.

In [5]:
class BatchedPatternSearch:
    """One iteration = one job: every candidate is known before any value comes back."""

    def __init__(self, x0, radius):
        self.x = np.array(x0, float)
        self.x_prev = np.array(x0, float)
        self.r = float(radius)

    def ask(self):
        cands = [self.x.copy()]
        for i in range(self.x.size):
            for sgn in (+1, -1):
                y = self.x.copy()
                y[i] += sgn * self.r
                cands.append(y)
        step = self.x - self.x_prev
        if np.linalg.norm(step) > 1e-9:
            cands.append(self.x + step)
        return np.array(cands)

    def tell(self, cands, values, tol):
        j = int(np.argmin(values))
        if j != 0 and values[j] < values[0] - tol:
            self.x_prev, self.x = self.x.copy(), cands[j].copy()
            self.r = min(self.r * 1.3, np.pi / 2)
            return True
        self.r = max(self.r * 0.6, 0.02)
        return False


def run_bps(x0, seed_tag, reserve):
    opt = BatchedPatternSearch(x0, RADIUS0)
    tr = dict(cvar=[], best_cvar=[], ideal_ratio=[], radius=[], moved=[], x=[], cand_cvar=[])
    best = dict(cvar=np.inf, x=np.array(x0, float))
    for it in range(N_JOBS):
        cands = opt.ask()
        shots_each = JOB_SHOTS // len(cands)
        try:
            dists = submit(isa_circuit, cands, shots_each, P_LAYERS, inst["N"], D2Q,
                           f"{seed_tag}_bps{it+1}", reserve)
        except BudgetStop as e:
            print(f"  budget stop after {it} batched jobs: {e}")
            break
        v = np.array([cvar(inst, d, ALPHA) for d in dists])
        tr["cand_cvar"].append(v.tolist())
        j = int(np.argmin(v))
        if v[j] < best["cvar"]:
            best.update(cvar=float(v[j]), x=cands[j].copy())
        p0 = dists[0]
        sd = np.sqrt(max(0.0, p0 @ inst["diag"] ** 2 - (inst["diag"] @ p0) ** 2))
        moved = opt.tell(cands, v, 0.5 * sd / np.sqrt(shots_each))
        tr["cvar"].append(float(v.min()))
        tr["best_cvar"].append(float(best["cvar"]))
        tr["ideal_ratio"].append(quality(inst, qaoa_probs(inst, opt.x, P_LAYERS))["ratio"])
        tr["radius"].append(float(opt.r))
        tr["moved"].append(bool(moved))
        tr["x"].append(opt.x.tolist())
        print(f"  bps {it+1:2d}/{N_JOBS}  CVaR {v.min():+.4f}  ideal ratio {tr['ideal_ratio'][-1]:.3f}")
    return tr, best


def run_serial(x0, seed_tag, reserve):
    tr = dict(values=[], best=[], x=[], ideal_ratio=[])
    best = dict(cvar=np.inf, x=np.array(x0, float))

    class _Stop(Exception):
        pass

    def f(x):
        if len(tr["values"]) >= N_JOBS:
            raise _Stop
        try:
            d = submit(isa_circuit, np.atleast_2d(x), JOB_SHOTS, P_LAYERS, inst["N"], D2Q,
                       f"{seed_tag}_ser{len(tr['values'])+1}", reserve)[0]
        except BudgetStop as e:
            print(f"  budget stop after {len(tr['values'])} serial jobs: {e}")
            raise _Stop
        v = cvar(inst, d, ALPHA)
        if v < best["cvar"]:
            best.update(cvar=float(v), x=np.array(x, float))
        tr["values"].append(float(v))
        tr["best"].append(float(best["cvar"]))
        tr["x"].append(list(map(float, x)))
        tr["ideal_ratio"].append(quality(inst, qaoa_probs(inst, np.array(x, float),
                                                          P_LAYERS))["ratio"])
        print(f"  ser {len(tr['values']):2d}/{N_JOBS}  CVaR {v:+.4f}  "
              f"ideal ratio {tr['ideal_ratio'][-1]:.3f}")
        return v

    try:
        minimize(f, x0, method="COBYLA", options={"maxiter": N_JOBS + 5, "rhobeg": RADIUS0})
    except _Stop:
        pass
    return tr, best

---
## Run the three starts

The JSON is rewritten after every start, so an interrupted run still leaves everything collected so far.

In [6]:
OUT = f"qaoa_jobs_replicates_{N_ASSETS}assets_{BACKEND_NAME}.json"
results = {}
k = np.arange(1, P_LAYERS + 1)

for dt in DT_STARTS:
    x0 = np.concatenate([dt * (1 - (k - 0.5) / P_LAYERS), -dt * k / P_LAYERS])
    tag = f"dt{dt:.2f}".replace(".", "")
    print(f"\n===== start dt = {dt}   x0 = {np.round(x0, 4)}   "
          f"noiseless CVaR {cvar(inst, qaoa_probs(inst, x0, P_LAYERS), ALPHA):+.4f}")

    # hold back what this start still owes: the serial arm plus two read-outs
    res_after_bps = N_JOBS * predict_qpu(JOB_SHOTS, D2Q) + 2 * predict_qpu(SHOTS_FINAL, D2Q)
    bps_tr, bps_best = run_bps(x0, tag, reserve=res_after_bps)
    ser_tr, ser_best = run_serial(x0, tag, reserve=2 * predict_qpu(SHOTS_FINAL, D2Q))

    reads = {}
    for nm, xb in (("bps", bps_best["x"]), ("serial", ser_best["x"])):
        try:
            reads[nm] = submit(isa_circuit, np.atleast_2d(xb), SHOTS_FINAL, P_LAYERS,
                               inst["N"], D2Q, f"{tag}_readout_{nm}", 0.0)[0].tolist()
        except BudgetStop as e:
            print(f"  no read-out for {nm}: {e}")

    results[f"{dt}"] = dict(
        dt=dt, x0=x0.tolist(),
        bps=dict(trace=bps_tr, best_x=bps_best["x"].tolist(), best_cvar=bps_best["cvar"]),
        serial=dict(trace=ser_tr, best_x=ser_best["x"].tolist(), best_cvar=ser_best["cvar"]),
        readouts=reads,
        final={nm: quality(inst, np.array(p)) for nm, p in reads.items()})

    with open(OUT, "w") as fh:
        json.dump(dict(provenance=dict(n_assets=N_ASSETS, budget=N_ASSETS // 2,
                                       backend=str(backend.name), p=P_LAYERS, shots=SHOTS,
                                       job_shots=JOB_SHOTS, n_jobs_per_arm=N_JOBS, alpha=ALPHA,
                                       radius0=RADIUS0, shots_final=SHOTS_FINAL,
                                       lam=inst["lam"], mu=inst["mu"],
                                       date=time.strftime("%Y-%m-%d %H:%M:%S"),
                                       chain=list(map(int, chain)), n2q=n2q, d2q=D2Q),
                       ledger=dict(jobs=n_jobs_used, shots=shots_used,
                                   qpu_seconds=qpu_used, log=job_log),
                       starts=results), fh)
    print(f"  saved {OUT}  ({n_jobs_used} jobs, {qpu_used:.1f} s so far)")

print(f"\nDONE: {n_jobs_used} jobs, {shots_used:,} shots, {qpu_used:.1f} s metered QPU")


===== start dt = 0.35   x0 = [ 0.2917  0.175   0.0583 -0.1167 -0.2333 -0.35  ]   noiseless CVaR -2.9019
    job   1 [dt035_bps1] id=da3resc3jnrc73afen00   8.33 qpu s (model  8.33)     16 s wall  (total    8.3 / 700 s)
  bps  1/10  CVaR -2.9496  ideal ratio 0.748
    job   2 [dt035_bps2] id=da3rf0botlns739a2h60   8.33 qpu s (model  8.33)     15 s wall  (total   16.7 / 700 s)
  bps  2/10  CVaR -3.2539  ideal ratio 0.885
    job   3 [dt035_bps3] id=da3rf443jnrc73afenb0   8.33 qpu s (model  8.33)     15 s wall  (total   25.0 / 700 s)
  bps  3/10  CVaR -3.3279  ideal ratio 0.830
    job   4 [dt035_bps4] id=da3rf861vhnc73fk7aog   8.33 qpu s (model  8.33)     17 s wall  (total   33.3 / 700 s)
  bps  4/10  CVaR -3.3616  ideal ratio 0.830
    job   5 [dt035_bps5] id=da3rfcc3jnrc73afeno0   8.33 qpu s (model  8.33)     16 s wall  (total   41.7 / 700 s)
  bps  5/10  CVaR -3.4904  ideal ratio 0.869
    job   6 [dt035_bps6] id=da3rfg43jnrc73afensg   8.33 qpu s (model  8.33)     16 s wall  (total   

In [7]:
# quick look: jobs each arm needs to reach 95% of its own final noiseless quality
print(f"{'start':>7}{'BPS job1':>10}{'BPS final':>11}{'BPS n95':>9}"
      f"{'SER job1':>10}{'SER final':>11}{'SER n95':>9}")
for key, r in results.items():
    b = np.maximum.accumulate(r["bps"]["trace"]["ideal_ratio"])
    s = np.maximum.accumulate(r["serial"]["trace"]["ideal_ratio"])
    nb_ = int(np.argmax(b >= 0.95 * b[-1])) + 1
    ns_ = int(np.argmax(s >= 0.95 * s[-1])) + 1
    print(f"{key:>7}{b[0]:>10.3f}{b[-1]:>11.3f}{nb_:>9d}{s[0]:>10.3f}{s[-1]:>11.3f}{ns_:>9d}")

  start  BPS job1  BPS final  BPS n95  SER job1  SER final  SER n95
   0.35     0.748      0.888        2     0.611      0.891        8
    0.6     0.878      0.920        1     0.788      0.919        4
   0.85     0.897      0.898        1     0.866      0.912        3
